# NHS England Maternal Care Pathways Master Pipeline
## Stage 5.2 - Load medical comorbidities data from MSDS, hospital spells, or A&E visits (now with diagnosis outcomes)

### Compiled by Ethan Phillips (Oxford) with code from Chuka Ezeoguine (LBS)

### Last updated 22.11.2025

In [0]:
import sys
import numpy as np
import pandas as pd
from functools import reduce
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR

%run "/Workspace/Shared/Helper_Methods_EP"

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name_1 = "msds_processed_cleaned_vars_stage"
df_master = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name_1}")

## Get LD site, start, and end

In [0]:
# Load data sources
SCHEMA = "dsa_712819_x8g2j_collab"

MSDS_BOOKING_TBL = f"{SCHEMA}.msds_v2_demographics_booking_and_pregnancy_all_years"  # orgsiteidintra
MSDS_BABY_DEMO_TBL = f"{SCHEMA}.msds_v2_baby_demographics_all_years"  # orgsiteidactual, settingplacebirth, merofbirthbaby
HES_APC = f"{SCHEMA}.hes_apc_all_years"
SITE_MAPPING = f"{SCHEMA}.nhs_trust_sites_mapping_ets"
PARENT_SITE_MAPPING = "reference_data.dss_corporate.org_daily"

baby_demographics_table = spark.table(MSDS_BABY_DEMO_TBL)
booking_table = spark.table(MSDS_BOOKING_TBL)
site_mapping_table = spark.table(SITE_MAPPING)
parent_site_mapping_table = spark.table(PARENT_SITE_MAPPING)

# From baby_demographics select and aggregate by UniqPregID:
# - fyear
# - OrgSiteIDActualDelivery
# - SettingPlaceBirth
# - UniqPregID
# - OrgCodeProvider
# - Baby year of birth, month of birth

fyear_bd_col        = pick_col(baby_demographics_table, ["fyear"])
setting_del_col  = pick_col(baby_demographics_table, ["SettingPlaceBirth"])
preg_bd_col         = pick_col(baby_demographics_table, ["UniqPregID"])
prov_col         = pick_col(baby_demographics_table, ["OrgCodeProvider"])
b_yob_col = pick_col(baby_demographics_table, ["yearofbirthbaby"])
b_mob_col = pick_col(baby_demographics_table, ["monthofbirthbaby"])
b_dow_col = pick_col(baby_demographics_table, ["dayofbirthbaby"])  # 01=Mon … 07=Sun

baby_demographics_table = baby_demographics_table.select(
    [preg_bd_col, fyear_bd_col, setting_del_col, prov_col, b_yob_col, b_mob_col, b_dow_col]
)

agg_bd_exprs = [
    F.last(F.col(prov_col), ignorenulls=True).alias("OrgCodeProvider"),
    F.last(F.col(setting_del_col),  ignorenulls=True).alias("SettingPlaceBirth"),
    F.max(F.col(fyear_bd_col)).alias("fyear_bd"),
    F.first(F.col(b_yob_col).cast("int"), ignorenulls=True).alias("y_birth"),
    F.first(F.col(b_mob_col).cast("int"), ignorenulls=True).alias("m_birth"),
    F.first(F.col(b_dow_col).cast("int"), ignorenulls=True).alias("dow_birth_code")
]

baby_demographics_condensed = baby_demographics_table.groupBy(F.col(preg_bd_col)).agg(*agg_bd_exprs)


# From booking select and aggregate by UniqPregID:
# - fyear
# - UniqPregID
# - OrgSiteIDIntra
# - SettingIntraCare
# - StartDateMotherDeliveryHospProvSpell
# - DischargeDateMotherHosp

fyear_bk_col      = pick_col(booking_table, ["fyear"])
site_intra_col    = pick_col(booking_table, ["OrgSiteIDIntra"])
setting_intra_col = pick_col(booking_table, ["SettingIntraCare"])
preg_bk_col          = pick_col(booking_table, ["UniqPregID"])
start_date_col    = pick_col(booking_table, ["StartDateMotherDeliveryHospProvSpell"])
end_date_col     = pick_col(booking_table, ["DischargeDateMotherHosp"])
alt_end_date_col = pick_col(booking_table, ["DischargeDateMatService"])

booking_table = booking_table.select([preg_bk_col, fyear_bk_col, site_intra_col, setting_intra_col, start_date_col, end_date_col, alt_end_date_col])

agg_bk_exprs = [
    F.max(F.col(fyear_bk_col)).alias("fyear_bk"),
    F.last(F.col(site_intra_col), ignorenulls=True).alias("OrgSiteIDIntra"),
    F.last(F.col(setting_intra_col),  ignorenulls=True).alias("SettingIntraCare"),
    F.last(F.col(start_date_col), ignorenulls=True).alias("StartDateMotherDeliveryHospProvSpell"),
    F.last(F.col(end_date_col),  ignorenulls=True).alias("DischargeDateMotherHosp"),
    F.last(F.col(alt_end_date_col),  ignorenulls=True).alias("DischargeDateMatService")
]

booking_condensed = booking_table.groupBy(F.col(preg_bk_col)).agg(*agg_bk_exprs)

In [0]:
df_master_updated = (
    df_master
    .select("UniqPregID", "Person_ID_DEID", "EPIKEY", "fyear", "ld_site_id", "ld_hosp_start_date", "labour_onset_date", "ld_disch_date")
    .join(
        baby_demographics_condensed, 
        on="UniqPregID",
        how="left"
    ).join(
            booking_condensed,
            on="UniqPregID",
            how="left"
    )
)

In [0]:
site_cols = ["ld_site_id", "OrgSiteIDIntra"]
setting_cols = ["SettingPlaceBirth", "SettingIntraCare"]
start_date_cols = ["ld_hosp_start_date", "labour_onset_date"]
end_date_cols = ["ld_disch_date", "DischargeDateMatService"]
fyear_cols = ["fyear", "fyear_bk", "fyear_bd"]
non_sites = ["Unknown", "N||SITE", "TESTCODE", "ZZ201", "ZZ203", "ZZ777", "ZZ888", "ZZ999", "89999", "89997"]

df_master_updated = (
    df_master_updated
    .withColumns({
        "OrgSiteIDLabourDelivery": get_partum_info(site_cols, non_sites),
        "SettingPlaceLabourDelivery": get_partum_info(setting_cols, []),
        "StartDateLabourDelivery": get_partum_info(start_date_cols, []),
        "EndDateLabourDelivery": get_partum_info(end_date_cols, []),
        "FyearPreg": get_partum_info(fyear_cols, [])
    })
)

df_master_updated = (
    df_master_updated
    .withColumn(
        "in_between_period", 
        F.datediff("EndDateLabourDelivery", "StartDateLabourDelivery")
    )
    .withColumn(
        "EndDateLabourDelivery", 
        when(
            (col("in_between_period") > 14),
            F.least(
                F.date_add(F.col("StartDateLabourDelivery"), 14),
                col("DischargeDateMatService")
            )
        ).otherwise(col("EndDateLabourDelivery"))
    )
    .withColumn("EndDateLabourDelivery", adjust_end_date("StartDateLabourDelivery", "EndDateLabourDelivery"))
    .withColumn("in_between_period", F.datediff("EndDateLabourDelivery", "StartDateLabourDelivery"))
)

In [0]:
# Join df_master_final with labour/delivery site name and parent name and ID
org_code_col    = pick_col(site_mapping_table, ["organisation_code"])
org_name_col    = pick_col(site_mapping_table, ["name"])
parent_code_col = pick_col(site_mapping_table, ["parent_organisation_code"])
parent_org_code_col = pick_col(parent_site_mapping_table, ["ORG_CODE"])
parent_org_name_col = pick_col(parent_site_mapping_table, ["NAME"])

site_mapping_table = site_mapping_table.select(org_code_col, org_name_col, parent_code_col).withColumnRenamed("name", "org_name")
parent_site_mapping_table = parent_site_mapping_table.select(parent_org_code_col, parent_org_name_col).withColumnRenamed("NAME", "parent_name")
mapping_table = (
    site_mapping_table
    .join(
        parent_site_mapping_table,
        site_mapping_table["parent_organisation_code"] == parent_site_mapping_table["ORG_CODE"],
        "left"
    )
    .select(
        F.col("organisation_code"),
        F.col("org_name"),
        F.col("parent_organisation_code"),
        F.col("parent_name")
    )
    .groupBy("organisation_code").agg(*[
        F.first("org_name", ignorenulls=True).alias("org_name"),
        F.first("parent_organisation_code", ignorenulls=True).alias("parent_organisation_code"),
        F.first("parent_name", ignorenulls=True).alias("parent_name"),
    ])
)

df_master_updated = (
    df_master_updated.join(
        mapping_table, 
        df_master_updated["OrgSiteIDLabourDelivery"] == mapping_table["organisation_code"], 
        how="left"
    )
    .withColumnRenamed("org_name", "OrgSiteLabourDelivery_name")
    .withColumnRenamed("parent_organisation_code", "OrgSiteLabourDelivery_parentID")
    .withColumnRenamed("parent_name", "OrgSiteLabourDelivery_parent_name")
    .drop("organisation_code")
)

In [0]:
df_master_updated = (
    df_master_updated
    .filter(
        col("OrgSiteIDLabourDelivery").isNotNull() & 
        col("StartDateLabourDelivery").isNotNull() & 
        col("EndDateLabourDelivery").isNotNull() & 
        (col("StartDateLabourDelivery") <=  col("EndDateLabourDelivery"))
    )
    .drop(*["ld_hosp_start_date"])
    .withColumnsRenamed({
        "OrgSiteIDLabourDelivery": "ld_hosp_org_site_id",
        "SettingPlaceLabourDelivery": "ld_hosp_org_site_setting_id",
        "StartDateLabourDelivery": "ld_hosp_start_date",
        "EndDateLabourDelivery": "ld_hosp_disch_date"
    })
    .select("UniqPregID", "ld_hosp_org_site_id", "ld_hosp_org_site_setting_id", "ld_hosp_start_date", "ld_hosp_disch_date")
)

In [0]:
write_table_name = "msds_ld_data"

df_master_updated.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")

## Find diagnoses and comorbidities

In [0]:
df_master_pregs = (
    df_master
    .drop(*['ld_hosp_start_date', 'ld_disch_date'])
    .join(df_master_updated, on="UniqPregID", how="left")
    .select("UniqPregID", "Person_ID_DEID", "last_period_date", "antenatal_appt_date", "ld_hosp_start_date", "ld_hosp_disch_date")
).withColumn(
    "last_period_date", when(col("last_period_date").isNull(), col("antenatal_appt_date")).otherwise(col("last_period_date"))
).drop("antenatal_appt_date")

In [0]:
exclusion_codes = [
    # ICD-10 - Preeclampsia / Gestational Hypertension
    "O14", "O14.0", "O14.1", "O14.2", "O14.9",
    "O11", "O15", "O15.0", "O15.1", "O15.2", "O15.9",
    "O10", "O13", "O16",

    # SNOMED - Preeclampsia
    "15938005", "29738008", "71701000119105", "72022006",
    "198992004", "169560008", "398254007", "427889009",
    "48194001", "69909000", "111479006"

    # ICD-10 - Gestational Diabetes
    "O24.4", "O24.410", "O24.414", "O24.419", "O24.42", "O24.43",
    
    # SNOMED - Gestational Diabetes
    "11687002", "73211009", "724714000", "724715004", "724716003"
]

hypertension_codes = [
    # ICD-10
    "I10", "I11", "I11.0", "I11.9", "I12", "I12.0", "I12.9", "I13", "I13.0", "I13.1", "I13.2",
    "I15", "I15.0", "I15.9", "I16", "I16.0", "I16.1", "I16.9", "I27.0", "I27.2",
    # SNOMED
    "38341003", "59621000", "70995007", "428575007", "428648006", "428755001",
    "697930005", "698296002", "371622005", "371125006", "48146000", "56218007"
]

preeclampsia_codes = [
    # ICD-10
    "O14", "O14.0", "O14.1", "O14.2", "O14.9", "O11", "O15", "O15.0", "O15.1", "O15.2", "O15.9",
    "O10", "O13", "O16",
    # SNOMED
    "15938005", "29738008", "71701000119105", "72022006", "198992004", "169560008",
    "398254007", "427889009", "48194001", "69909000", "111479006"
]

gestational_diabetes_codes = [
    # ICD-10
    "O24.4", "O24.410", "O24.414", "O24.419", "O24.42", "O24.43", "O24.9"
    
    # SNOMED
    "11687002", "73211009", "724714000", "724715004", "724716003"
]

####Get diagnoses from ECDS

In [0]:
df_ecds = spark.sql("SELECT DISTINCT PERSON_ID_DEID, fyear, ARRIVAL_DATE, COMORBIDITIES_1, COMORBIDITIES_2, COMORBIDITIES_3, COMORBIDITIES_4, COMORBIDITIES_5, COMORBIDITIES_6, COMORBIDITIES_7, COMORBIDITIES_8, COMORBIDITIES_9, COMORBIDITIES_10, DIAGNOSIS_CODE_1, DIAGNOSIS_CODE_2, DIAGNOSIS_CODE_3, DIAGNOSIS_CODE_4, DIAGNOSIS_CODE_5, DIAGNOSIS_CODE_6, DIAGNOSIS_CODE_7, DIAGNOSIS_CODE_8, DIAGNOSIS_CODE_9, DIAGNOSIS_CODE_10 FROM `dsa_712819_x8g2j`.`dsa_712819_x8g2j_collab`.`lowlat_ecds_all_years`;")
print(f"Number of rows: {'{:,}'.format(df_ecds.count())}. Number of columns: {'{:,}'.format(len(df_ecds.columns))}")
assert df_ecds.groupBy(df_ecds.columns).count().filter("count > 1").count() == 0


In [0]:
# Join A&E data to pregnancies

df_ecds_diags = df_ecds.join(df_master_pregs, on="Person_ID_DEID", how="inner")

# Filter to visits before LMP
df_ecds_diags_pre = df_ecds_diags.filter(col("Arrival_Date") < col("last_period_date"))

# Filter to visits during pregnancy
df_ecds_diags_during = df_ecds_diags.filter((col("Arrival_Date") > col("last_period_date")) & (col("Arrival_Date") < col("ld_hosp_disch_date")))

# Group by pregnancy and collect diagnosis/comorbidity/arrival lists
def collect_ecds_diags(df, suffix):
    df = (
        df
        .groupBy("UniqPregID")
        .agg(
            collect_set("DIAGNOSIS_CODE_1").alias(f"DiagnosisCodes_{suffix}_1"),
            collect_set("DIAGNOSIS_CODE_2").alias(f"DiagnosisCodes_{suffix}_2"),
            collect_set("DIAGNOSIS_CODE_3").alias(f"DiagnosisCodes_{suffix}_3"),
            collect_set("DIAGNOSIS_CODE_4").alias(f"DiagnosisCodes_{suffix}_4"),
            collect_set("DIAGNOSIS_CODE_5").alias(f"DiagnosisCodes_{suffix}_5"),
            collect_set("DIAGNOSIS_CODE_6").alias(f"DiagnosisCodes_{suffix}_6"),
            collect_set("DIAGNOSIS_CODE_7").alias(f"DiagnosisCodes_{suffix}_7"),
            collect_set("DIAGNOSIS_CODE_8").alias(f"DiagnosisCodes_{suffix}_8"),
            collect_set("DIAGNOSIS_CODE_9").alias(f"DiagnosisCodes_{suffix}_9"),
            collect_set("DIAGNOSIS_CODE_10").alias(f"DiagnosisCodes_{suffix}_10"),
            collect_set("COMORBIDITIES_1").alias(f"Comorbidities_{suffix}_1"),
            collect_set("COMORBIDITIES_2").alias(f"Comorbidities_{suffix}_2"),
            collect_set("COMORBIDITIES_3").alias(f"Comorbidities_{suffix}_3"),
            collect_set("COMORBIDITIES_4").alias(f"Comorbidities_{suffix}_4"),
            collect_set("COMORBIDITIES_5").alias(f"Comorbidities_{suffix}_5"),
            collect_set("COMORBIDITIES_6").alias(f"Comorbidities_{suffix}_6"),
            collect_set("COMORBIDITIES_7").alias(f"Comorbidities_{suffix}_7"),
            collect_set("COMORBIDITIES_8").alias(f"Comorbidities_{suffix}_8"),
            collect_set("COMORBIDITIES_9").alias(f"Comorbidities_{suffix}_9"),
            collect_set("COMORBIDITIES_10").alias(f"Comorbidities_{suffix}_10"),
            collect_set("ARRIVAL_DATE").alias(f"ArrivalDates_{suffix}")
        )
    )

    # Collect all diagnosis and comorbidity cols 
    diagnosis_cols = [f"DiagnosisCodes_{suffix}_{i}" for i in range(1, 11)]
    comorbidity_cols = [f"Comorbidities_{suffix}_{i}" for i in range(1, 11)]

    diagnosis_union = col(diagnosis_cols[0])
    for col_name in diagnosis_cols[1:]:
        diagnosis_union = array_union(diagnosis_union, col(col_name))

    comorbidity_union = col(comorbidity_cols[0])
    for col_name in comorbidity_cols[1:]:
        comorbidity_union = array_union(comorbidity_union, col(col_name))

    df = df.withColumns({
        f"DiagnosisCodes_{suffix}": diagnosis_union,
        f"Comorbidities_{suffix}": comorbidity_union
    }).select(
        f"DiagnosisCodes_{suffix}", f"Comorbidities_{suffix}", "UniqPregID"
    )

    return df

df_ecds_diags_pre = collect_ecds_diags(df_ecds_diags_pre, "PrePreg")
df_ecds_diags_during = collect_ecds_diags(df_ecds_diags_during, "DuringPreg")



In [0]:
ecds_diag_desc = spark.table("dsa_712819_x8g2j.dsa_712819_x8g2j_collab.ecdsdescriptionsdiag")
ecds_diag_desc= ecds_diag_desc.select(
    col("diagnosis_code_snomed").cast("string").alias("diag_code"),
    col("higher_level_description").alias("diag_category")
)

ecds_diag_desc = ecds_diag_desc.withColumn(
    "diag_code",
    regexp_replace(col("diag_code").cast("string"), r"\.0$", "")
).select(
    "diag_code",
    "diag_category"
)

# Collect lookup as dict
code_to_desc = {row['diag_code']: row['diag_category'] for row in ecds_diag_desc.collect()}

# Define UDF
def map_codes_to_desc(codes):
    if codes is None:
        return None
    return [code_to_desc.get(c, f"Unknown({c})") for c in codes]

map_codes_udf = udf(map_codes_to_desc, ArrayType(StringType()))

# Apply UDF
df_ecds_diags_pre = df_ecds_diags_pre.withColumn(
    "DiagnosisDescriptions_PrePreg",
    map_codes_udf("DiagnosisCodes_PrePreg")
)

df_ecds_diags_during = df_ecds_diags_during.withColumn(
    "DiagnosisDescriptions_DuringPreg",
    map_codes_udf("DiagnosisCodes_DuringPreg")
)

In [0]:
ecds_comorb_desc = spark.table("dsa_712819_x8g2j.dsa_712819_x8g2j_collab.ecdsdescriptionscomorb")
ecds_comorb_desc= ecds_comorb_desc.select(
    col("comorbidity_code").cast("string").alias("comorb_code"),
    col("higher_level_description").alias("comorb_category")
)

ecds_comorb_desc = ecds_comorb_desc.withColumn(
    "comorb_code",
    regexp_replace(col("comorb_code").cast("string"), r"\.0$", "")
).select(
    "comorb_code",
    "comorb_category"
)

# Convert lookup table to dict
comorb_to_desc = {row['comorb_code']: row['comorb_category'] for row in ecds_comorb_desc.collect()}

# Define UDF
def map_comorbs_to_desc(codes):
    if codes is None:
        return None
    return [comorb_to_desc.get(c, f"Unknown({c})") for c in codes]

map_comorbs_udf = udf(map_comorbs_to_desc, ArrayType(StringType()))

# Apply UDF
df_ecds_diags_pre = df_ecds_diags_pre.withColumn(
    "ComorbidityDescriptions_PrePreg",
    map_comorbs_udf("Comorbidities_PrePreg")
)

df_ecds_diags_during = df_ecds_diags_during.withColumn(
    "ComorbidityDescriptions_DuringPreg",
    map_comorbs_udf("Comorbidities_DuringPreg")
)

In [0]:
df_ecds_diags_pre = df_ecds_diags_pre.withColumns({
    "DiagnosisCodes_PrePreg_ECDS": col("DiagnosisCodes_PrePreg"),
    "DiagnosisDescriptions_PrePreg_ECDS": col("DiagnosisDescriptions_PrePreg"),
    "Comorbidities_PrePreg_ECDS": col("Comorbidities_PrePreg"),
    "ComorbidityDescriptions_PrePreg_ECDS": col("ComorbidityDescriptions_PrePreg")
}).select(
    "UniqPregID", 
    "DiagnosisCodes_PrePreg_ECDS", "DiagnosisDescriptions_PrePreg_ECDS",
    "Comorbidities_PrePreg_ECDS", "ComorbidityDescriptions_PrePreg_ECDS"
)

df_ecds_diags_during = df_ecds_diags_during.withColumns({
    "DiagnosisCodes_DuringPreg_ECDS": col("DiagnosisCodes_DuringPreg"),
    "DiagnosisDescriptions_DuringPreg_ECDS": col("DiagnosisDescriptions_DuringPreg"),
    "Comorbidities_DuringPreg_ECDS": col("Comorbidities_DuringPreg"),
    "ComorbidityDescriptions_DuringPreg_ECDS": col("ComorbidityDescriptions_DuringPreg")
}).select(
    "UniqPregID", 
    "DiagnosisCodes_DuringPreg_ECDS", "DiagnosisDescriptions_DuringPreg_ECDS",
    "Comorbidities_DuringPreg_ECDS", "ComorbidityDescriptions_DuringPreg_ECDS"
)

df_ecds_diags = df_ecds_diags_pre.join(df_ecds_diags_during, on="UniqPregID", how="outer")

####Get diagnoses from MSDS hosp spells (prior to current pregnancy and during)

In [0]:
df_msds_diags = spark.sql(
    """
    SELECT DISTINCT person_id_mother_deid, diagnosisandhistory, diagnosisdate,mastericd10diagnosisandhistorycode, mastericd10diagnosisandhistorydesc, mastersnomedctdiagnosisandhistorycode, mastersnomedctdiagnosisandhistoryterm, diagnosisscheme FROM `dsa_712819_x8g2j`.`dsa_712819_x8g2j_collab`.`msds_v2_diagnoses_and_history_all_years`;
    """
)

print(f"Number of rows: {'{:,}'.format(df_msds_diags.count())}. Number of columns: {'{:,}'.format(len(df_msds_diags.columns))}")
assert df_msds_diags.groupBy(df_msds_diags.columns).count().filter("count > 1").count() == 0

df_msds_diags = df_msds_diags.select(
    "person_id_mother_deid",
    "diagnosisandhistory",
    "diagnosisdate"
).withColumnRenamed("person_id_mother_deid", "person_id_deid")

# filter for patients in df_master
df_msds_diags_pre = df_msds_diags.join(
    df_master_pregs, on="person_id_deid", how="inner"
).filter(
    (col("diagnosisdate") < col("last_period_date"))
).groupBy("UniqPregID").agg(
    collect_set("diagnosisandhistory").alias("DiagnosisCodes_PrePreg_MSDS")
)

df_msds_diags_during = df_msds_diags.join(
    df_master_pregs, on="person_id_deid", how="inner"
).filter(
    (col("diagnosisdate") < col("ld_hosp_disch_date")) & \
    (col("diagnosisdate") > col("last_period_date"))
).groupBy("UniqPregID").agg(
    collect_set("diagnosisandhistory").alias("DiagnosisCodes_DuringPreg_MSDS")
)

In [0]:
df_msdsdesc = spark.sql("""
    SELECT DISTINCT 
        diagnosis_code_snomed,
        icd10,
        higher_level_description
    FROM dsa_712819_x8g2j_collab.msdsdescriptions
""")

# Update `higher_level_description` based on match
df_msdsdesc_updated = df_msdsdesc.withColumn(
    "higher_level_description",
    when(col("icd10").isin(preeclampsia_codes) | col("diagnosis_code_snomed").isin(preeclampsia_codes), "Preeclampsia")
    .when((col("icd10").isin(gestational_diabetes_codes)) | (col("diagnosis_code_snomed").isin(gestational_diabetes_codes)), "Gestational Diabetes")
    .otherwise(col("higher_level_description"))
)

In [0]:
# Collect mapping of codes to descriptions as dict
msds_desc_rows = df_msdsdesc_updated.select("diagnosis_code_snomed", "icd10", "higher_level_description").collect()

code_to_desc = {}
for row in msds_desc_rows:
    if row["diagnosis_code_snomed"]:
        code_to_desc[row["diagnosis_code_snomed"]] = row["higher_level_description"]
    if row["icd10"]:
        code_to_desc[row["icd10"]] = row["higher_level_description"]

# Define UDF
def map_msds_codes_to_desc(codes):
    if codes is None:
        return None
    return [code_to_desc.get(c, f"Unknown({c})") for c in codes]

map_msds_udf = udf(map_msds_codes_to_desc, ArrayType(StringType()))

# Apply UDF
df_msds_diags_pre = df_msds_diags_pre.withColumn(
    "DiagnosisDescriptions_PrePreg_MSDS",
    map_msds_udf("DiagnosisCodes_PrePreg_MSDS")
).select(
    "UniqPregID", 
    "DiagnosisCodes_PrePreg_MSDS", "DiagnosisDescriptions_PrePreg_MSDS"
)

df_msds_diags_during = df_msds_diags_during.withColumn(
    "DiagnosisDescriptions_DuringPreg_MSDS",
    map_msds_udf("DiagnosisCodes_DuringPreg_MSDS")
).select(
    "UniqPregID", 
    "DiagnosisCodes_DuringPreg_MSDS", "DiagnosisDescriptions_DuringPreg_MSDS"
)

df_msds_diags = df_msds_diags_pre.join(df_msds_diags_during, on="UniqPregID", how="outer")

####Get diagnoses from HES APC

In [0]:
# Load hosp diagnosis data
df_apc_diags = spark.sql(
"""
    SELECT PERSON_ID_DEID, ADMIDATE, DIAG_3_CONCAT, DIAG_4_CONCAT
    FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.hes_apc_all_years
"""
)


# Join to pregnancies in df_master and filter for timing before lmp
df_apc_diags = df_apc_diags.join(
    df_master_pregs,
    on="Person_ID_DEID",
    how="inner"
)

df_apc_diags_pre = df_apc_diags.filter(col("ADMIDATE") < col("last_period_date"))
df_apc_diags_during = df_apc_diags.filter((col("ADMIDATE") < col("ld_hosp_disch_date")) & (col("ADMIDATE") > col("last_period_date")))


In [0]:
def process_apc_diags(df, suffix):
    # Group and collect diagnosis lists
    df = (
        df
        .groupBy("UniqPregID") 
        .agg(
            collect_list("DIAG_3_CONCAT").alias(f"Diag3Codes_{suffix}_APC"),
            collect_list("DIAG_4_CONCAT").alias(f"Diag4Codes_{suffix}_APC")
        )
    )

    df = df.withColumn(
        f"Diag3Codes_{suffix}_APC",
        F.array_distinct(
            F.filter(
                F.flatten(
                    F.transform(
                        col(f"Diag3Codes_{suffix}_APC"),
                        lambda x: F.split(x, ",")
                    )
                ),
                lambda y: y != ""
            )
        )
    ).withColumn(
        f"Diag4Codes_{suffix}_APC",
        F.array_distinct(
            F.filter(
                F.flatten(
                    F.transform(
                        col(f"Diag4Codes_{suffix}_APC"),
                        lambda x: F.split(x, ",")
                    )
                ),
                lambda y: y != ""
            )
        )
    )

    return df

df_apc_diags_pre = process_apc_diags(df_apc_diags_pre, "PrePreg")
df_apc_diags_during = process_apc_diags(df_apc_diags_during, "DuringPreg")

In [0]:
df_first = spark.sql("SELECT * FROM dsa_712819_x8g2j_collab.apcdescriptionsfirst_cleaned")
df_second = spark.sql("SELECT * FROM dsa_712819_x8g2j_collab.apcdescriptionssecond")

# Build Code → Description Dictionary
apc_desc_dict = {}

def extract_code_mappings(row, df_cols):
    for col in df_cols:
        if col.startswith("code"):
            icd_agg_col = col.replace("code", "icd_agg_desc")
            if icd_agg_col in df_cols:
                code_val = row[col]
                agg_desc_val = row[icd_agg_col]
                if code_val and agg_desc_val:
                    apc_desc_dict[code_val] = agg_desc_val

first_cols = df_first.columns
for row in df_first.collect():
    extract_code_mappings(row, first_cols)

second_cols = df_second.columns
for row in df_second.collect():
    extract_code_mappings(row, second_cols)

# Define mapping function 
def map_to_agg_desc(codes):
    if codes is None:
        return None
    result = []
    for c in codes:
        if c in preeclampsia_codes:
            result.append("Preeclampsia")
        elif c in gestational_diabetes_codes:
            result.append("Gestational Diabetes")
        else:
            result.append(apc_desc_dict.get(c, f"Unknown({c})"))
    return result

map_agg_desc_udf = udf(map_to_agg_desc, ArrayType(StringType()))

df_apc_diags_pre = df_apc_diags_pre.withColumns({
    "Diag3AggDescriptions_PrePreg_APC": map_agg_desc_udf("Diag3Codes_PrePreg_APC"),
    "Diag4AggDescriptions_PrePreg_APC": map_agg_desc_udf("Diag4Codes_PrePreg_APC")
}).select(
    "UniqPregID",
    "Diag3Codes_PrePreg_APC", "Diag3AggDescriptions_PrePreg_APC",
    "Diag4Codes_PrePreg_APC", "Diag4AggDescriptions_PrePreg_APC"
)

df_apc_diags_during = df_apc_diags_during.withColumns({
    "Diag3AggDescriptions_DuringPreg_APC": map_agg_desc_udf("Diag3Codes_DuringPreg_APC"),
    "Diag4AggDescriptions_DuringPreg_APC": map_agg_desc_udf("Diag4Codes_DuringPreg_APC")
}).select(
    "UniqPregID",
    "Diag3Codes_DuringPreg_APC", "Diag3AggDescriptions_DuringPreg_APC",
    "Diag4Codes_DuringPreg_APC", "Diag4AggDescriptions_DuringPreg_APC"
)

df_apc_diags = df_apc_diags_pre.join(df_apc_diags_during, on="UniqPregID", how="outer")

####Get Diagnoses from HES A&E

In [0]:
# Define columns
diag2_cols = [
    "DIAG2_01", "DIAG2_02", "DIAG2_03", "DIAG2_04",
    "DIAG2_05", "DIAG2_06", "DIAG2_07", "DIAG2_08",
    "DIAG2_09", "DIAG2_10", "DIAG2_11", "DIAG2_12"
]
diag3_cols = [
    "DIAG3_01", "DIAG3_02", "DIAG3_03", "DIAG3_04",
    "DIAG3_05", "DIAG3_06", "DIAG3_07", "DIAG3_08",
    "DIAG3_09", "DIAG3_10", "DIAG3_11", "DIAG3_12"
]
diag_ae_cols = ["PERSON_ID_DEID", "ARRIVALDATE"] + diag2_cols + diag3_cols

# Load A&E data
df_hes_ae = spark.sql("""
    SELECT *
    FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.hes_ae_all_years
""")
df_ae_diags = df_hes_ae.select(*diag_ae_cols)

# Join to master preg list
df_ae_diags = df_ae_diags.join(
    df_master_pregs,
    on="Person_ID_DEID",
    how="inner"
)

df_ae_diags_pre = df_ae_diags.filter(
    col("ARRIVALDATE") < col("last_period_date")
)

df_ae_diags_during = df_ae_diags.filter(
    (col("ARRIVALDATE") < col("ld_hosp_disch_date")) & \
    (col("ARRIVALDATE") > col("last_period_date"))
)

In [0]:
def process_ae_diags(df, suffix):
    # Group and collect diagnosis lists
    df = (
        df
        .groupBy("UniqPregID")
        .agg(
            collect_list("DIAG2_01").alias(f"Diag2Codes_01_{suffix}_AE"),
            collect_list("DIAG2_02").alias(f"Diag2Codes_02_{suffix}_AE"),
            collect_list("DIAG2_03").alias(f"Diag2Codes_03_{suffix}_AE"),
            collect_list("DIAG2_04").alias(f"Diag2Codes_04_{suffix}_AE"),
            collect_list("DIAG2_05").alias(f"Diag2Codes_05_{suffix}_AE"),
            collect_list("DIAG2_06").alias(f"Diag2Codes_06_{suffix}_AE"),
            collect_list("DIAG2_07").alias(f"Diag2Codes_07_{suffix}_AE"),
            collect_list("DIAG2_08").alias(f"Diag2Codes_08_{suffix}_AE"),
            collect_list("DIAG2_09").alias(f"Diag2Codes_09_{suffix}_AE"),
            collect_list("DIAG2_10").alias(f"Diag2Codes_10_{suffix}_AE"),
            collect_list("DIAG2_11").alias(f"Diag2Codes_11_{suffix}_AE"),
            collect_list("DIAG2_12").alias(f"Diag2Codes_12_{suffix}_AE"),

            collect_list("DIAG3_01").alias(f"Diag3Codes_01_{suffix}_AE"),
            collect_list("DIAG3_02").alias(f"Diag3Codes_02_{suffix}_AE"),
            collect_list("DIAG3_03").alias(f"Diag3Codes_03_{suffix}_AE"),
            collect_list("DIAG3_04").alias(f"Diag3Codes_04_{suffix}_AE"),
            collect_list("DIAG3_05").alias(f"Diag3Codes_05_{suffix}_AE"),
            collect_list("DIAG3_06").alias(f"Diag3Codes_06_{suffix}_AE"),
            collect_list("DIAG3_07").alias(f"Diag3Codes_07_{suffix}_AE"),
            collect_list("DIAG3_08").alias(f"Diag3Codes_08_{suffix}_AE"),
            collect_list("DIAG3_09").alias(f"Diag3Codes_09_{suffix}_AE"),
            collect_list("DIAG3_10").alias(f"Diag3Codes_10_{suffix}_AE"),
            collect_list("DIAG3_11").alias(f"Diag3Codes_11_{suffix}_AE"),
            collect_list("DIAG3_12").alias(f"Diag3Codes_12_{suffix}_AE")
        ) 
        .withColumns({
            f"Diag2Codes_{suffix}_AE": F.array_distinct(
                F.flatten(F.array(
                    col(f"Diag2Codes_01_{suffix}_AE"),
                    col(f"Diag2Codes_02_{suffix}_AE"),
                    col(f"Diag2Codes_03_{suffix}_AE"),
                    col(f"Diag2Codes_04_{suffix}_AE"),
                    col(f"Diag2Codes_05_{suffix}_AE"),
                    col(f"Diag2Codes_06_{suffix}_AE"),
                    col(f"Diag2Codes_07_{suffix}_AE"),
                    col(f"Diag2Codes_08_{suffix}_AE"),
                    col(f"Diag2Codes_09_{suffix}_AE"),
                    col(f"Diag2Codes_10_{suffix}_AE"),
                    col(f"Diag2Codes_11_{suffix}_AE"),
                    col(f"Diag2Codes_12_{suffix}_AE")
                ))
            ),
            f"Diag3Codes_{suffix}_AE": F.array_distinct(
                F.flatten(F.array(
                    col(f"Diag3Codes_01_{suffix}_AE"),
                    col(f"Diag3Codes_02_{suffix}_AE"),
                    col(f"Diag3Codes_03_{suffix}_AE"),
                    col(f"Diag3Codes_04_{suffix}_AE"),
                    col(f"Diag3Codes_05_{suffix}_AE"),
                    col(f"Diag3Codes_06_{suffix}_AE"),
                    col(f"Diag3Codes_07_{suffix}_AE"),
                    col(f"Diag3Codes_08_{suffix}_AE"),
                    col(f"Diag3Codes_09_{suffix}_AE"),
                    col(f"Diag3Codes_10_{suffix}_AE"),
                    col(f"Diag3Codes_11_{suffix}_AE"),
                    col(f"Diag3Codes_12_{suffix}_AE")
                ))
            )
        }) 
        .select("UniqPregID", f"Diag2Codes_{suffix}_AE", f"Diag3Codes_{suffix}_AE")
    )

    return df

df_ae_diags_pre = process_ae_diags(df_ae_diags_pre, "PrePreg")
df_ae_diags_during = process_ae_diags(df_ae_diags_during, "DuringPreg")


In [0]:
# Load lookup
df_lookup = spark.sql("SELECT * FROM dsa_712819_x8g2j_collab.aedescriptions")

# Create dictionary
lookup_dict = {}

for row in df_lookup.collect():
    code_val = row['code']
    desc_val = row['higher_level_description']
    if code_val and desc_val:
        lookup_dict[str(code_val)] = desc_val

# Define UDF
def map_ae_codes(code_list):
    if code_list is None:
        return None
    return [lookup_dict.get(str(c), f"Unknown({c})") for c in code_list]

map_ae_udf = udf(map_ae_codes, ArrayType(StringType()))

# Apply and add cols
df_ae_diags_pre = df_ae_diags_pre.withColumns({
    "Diag2Descriptions_PrePreg_AE": map_ae_udf("Diag2Codes_PrePreg_AE"),
    "Diag3Descriptions_PrePreg_AE": map_ae_udf("Diag3Codes_PrePreg_AE")
}).select(
    "UniqPregID", 
    "Diag2Codes_PrePreg_AE", "Diag2Descriptions_PrePreg_AE",
    "Diag3Codes_PrePreg_AE", "Diag3Descriptions_PrePreg_AE"
)

df_ae_diags_during = df_ae_diags_during.withColumns({
    "Diag2Descriptions_DuringPreg_AE": map_ae_udf("Diag2Codes_DuringPreg_AE"),
    "Diag3Descriptions_DuringPreg_AE": map_ae_udf("Diag3Codes_DuringPreg_AE")
}).select(
    "UniqPregID", 
    "Diag2Codes_DuringPreg_AE", "Diag2Descriptions_DuringPreg_AE",
    "Diag3Codes_DuringPreg_AE", "Diag3Descriptions_DuringPreg_AE"
)

df_ae_diags = df_ae_diags_pre.join(df_ae_diags_during, on="UniqPregID", how="outer")

In [0]:
df_all_diags = (
  df_ecds_diags
  .join(df_msds_diags, on="UniqPregID", how="outer")
  .join(df_apc_diags, on="UniqPregID", how="outer")
  .join(df_ae_diags, on="UniqPregID", how="outer")
).withColumn(
  "DiagDescriptions_PrePreg_ALL",
  F.array(
    when(col("DiagnosisDescriptions_PrePreg_ECDS").isNotNull(), col("DiagnosisDescriptions_PrePreg_ECDS")).otherwise(["missing"]),
    when(col("ComorbidityDescriptions_PrePreg_ECDS").isNotNull(), col("ComorbidityDescriptions_PrePreg_ECDS")).otherwise(["missing"]),
    when(col("DiagnosisDescriptions_PrePreg_MSDS").isNotNull(), col("DiagnosisDescriptions_PrePreg_MSDS")).otherwise(["missing"]),
    when(col("Diag3AggDescriptions_PrePreg_APC").isNotNull(), col("Diag3AggDescriptions_PrePreg_APC")).otherwise(["missing"]),
    when(col("Diag4AggDescriptions_PrePreg_APC").isNotNull(), col("Diag4AggDescriptions_PrePreg_APC")).otherwise(["missing"]),
    when(col("Diag2Descriptions_PrePreg_AE").isNotNull(), col("Diag2Descriptions_PrePreg_AE")).otherwise(["missing"]),
    when(col("Diag3Descriptions_PrePreg_AE").isNotNull(), col("Diag3Descriptions_PrePreg_AE")).otherwise(["missing"])
  )
).withColumn(
  "DiagDescriptions_PrePreg_ALL",
  F.array_distinct(
    F.flatten(col("DiagDescriptions_PrePreg_ALL"))
  )
).withColumn(
  "DiagDescriptions_DuringPreg_ALL",
  F.array(
    when(col("DiagnosisDescriptions_DuringPreg_ECDS").isNotNull(), col("DiagnosisDescriptions_DuringPreg_ECDS")).otherwise(["missing"]),
    when(col("ComorbidityDescriptions_DuringPreg_ECDS").isNotNull(), col("ComorbidityDescriptions_DuringPreg_ECDS")).otherwise(["missing"]),
    when(col("DiagnosisDescriptions_DuringPreg_MSDS").isNotNull(), col("DiagnosisDescriptions_DuringPreg_MSDS")).otherwise(["missing"]),
    when(col("Diag3AggDescriptions_DuringPreg_APC").isNotNull(), col("Diag3AggDescriptions_DuringPreg_APC")).otherwise(["missing"]),
    when(col("Diag4AggDescriptions_DuringPreg_APC").isNotNull(), col("Diag4AggDescriptions_DuringPreg_APC")).otherwise(["missing"]),
    when(col("Diag2Descriptions_DuringPreg_AE").isNotNull(), col("Diag2Descriptions_DuringPreg_AE")).otherwise(["missing"]),
    when(col("Diag3Descriptions_DuringPreg_AE").isNotNull(), col("Diag3Descriptions_DuringPreg_AE")).otherwise(["missing"])
  )
).withColumn(
  "DiagDescriptions_DuringPreg_ALL",
  F.array_distinct(
    F.flatten(col("DiagDescriptions_DuringPreg_ALL"))
  )
).select(
  "UniqPregID", 
  "DiagDescriptions_PrePreg_ALL",
  "DiagDescriptions_DuringPreg_ALL"
)

In [0]:
# List of categories
categories = [
    "Preeclampsia",
    "Gestational Diabetes",
    "Certain infectious and parasitic diseases",
    "Neoplasms",
    "Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism",
    "Endocrine, nutritional and metabolic diseases",
    "Mental and behavioural disorders",
    "Diseases of the nervous system",
    "Diseases of the eye and adnexa",
    "Diseases of the ear and mastoid process",
    "Diseases of the circulatory system",
    "Diseases of the respiratory system",
    "Diseases of the digestive system",
    "Diseases of the skin and subcutaneous tissue",
    "Diseases of the musculoskeletal system and connective tissue",
    "Diseases of the genitourinary system",
    "Pregnancy with abortive outcome",
    "Oedema, proteinuria and hypertensive disorders in pregnancy, childbirth and the puerperium",
    "Other maternal disorders predominantly related to pregnancy",
    "Diabetes mellitus in pregnancy",
    "Maternal infectious and parasitic diseases classifiable elsewhere but complicating pregnancy, childbirth and the puerperium",
    "Certain conditions originating in the perinatal period",
    "Congenital malformations, deformations and chromosomal abnormalities",
    "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified",
    "Injury, poisoning and certain other consequences of external causes",
    "Provisional assignment of new diseases of uncertain etiology or emergency use",
    "Resistance to antimicrobial and antineoplastic drugs",
    "External causes of morbidity and mortality",
    "Factors influencing health status and contact with health services"
]

# Create binary columns for each category 
for cat in categories:
    col_name = cat.replace(" ", "_") \
                  .replace(",", "") \
                  .replace("(", "") \
                  .replace(")", "") \
                  .replace("/", "_") \
                  .replace("-", "_") \
                  .replace("__", "_") 

    col_name_pre = col_name + "_PrePreg"
    col_name_during = col_name + "_DuringPreg"
    
    df_all_diags = df_all_diags.withColumn(
        col_name_pre,
        when(array_contains(col("DiagDescriptions_PrePreg_ALL"), cat), F.lit(1)).otherwise(F.lit(0))
    ).withColumn(
        col_name_during,
        when(array_contains(col("DiagDescriptions_DuringPreg_ALL"), cat), F.lit(1)).otherwise(F.lit(0))
    )

df_all_diags = df_all_diags.withColumns({
    "Preeclampsia_PrePreg": when((col("Preeclampsia_PrePreg")==1) | (col("Oedema_proteinuria_and_hypertensive_disorders_in_pregnancy_childbirth_and_the_puerperium_PrePreg")==1), 1).otherwise(0),
    "Gestational_Diabetes_PrePreg": when((col("Diabetes_mellitus_in_pregnancy_PrePreg")==1) | (col("Gestational_Diabetes_PrePreg")==1), 1).otherwise(0),
    "Preeclampsia_DuringPreg": when((col("Preeclampsia_DuringPreg")==1) | (col("Oedema_proteinuria_and_hypertensive_disorders_in_pregnancy_childbirth_and_the_puerperium_DuringPreg")==1), 1).otherwise(0),
    "Gestational_Diabetes_DuringPreg": when((col("Diabetes_mellitus_in_pregnancy_DuringPreg")==1) | (col("Gestational_Diabetes_DuringPreg")==1), 1).otherwise(0),
}).drop(*[
    "Oedema_proteinuria_and_hypertensive_disorders_in_pregnancy_childbirth_and_the_puerperium_PrePreg",
    "Oedema_proteinuria_and_hypertensive_disorders_in_pregnancy_childbirth_and_the_puerperium_DuringPreg",
    "Diabetes_mellitus_in_pregnancy_PrePreg",
    "Diabetes_mellitus_in_pregnancy_DuringPreg"
]).withColumnsRenamed({
    "Preeclampsia_PrePreg": "Prior_Preeclampsia",
    "Gestational_Diabetes_PrePreg": "Prior_Gestational_Diabetes",

    "Preeclampsia_DuringPreg": "Preeclampsia_this_pregnancy",
    "Gestational_Diabetes_DuringPreg": "Gestational_Diabetes_this_pregnancy",
    
    "Certain_infectious_and_parasitic_diseases_PrePreg": "Prior_Infectious_Disease",
    "Neoplasms_PrePreg": "Prior_Neoplasm",
    "Diseases_of_the_blood_and_blood_forming_organs_and_certain_disorders_involving_the_immune_mechanism_PrePreg": "Prior_Blood_or_Immune_Disease",
    "Endocrine_nutritional_and_metabolic_diseases_PrePreg": "Prior_Endocrine_or_Metabolic_Disease",
    "Mental_and_behavioural_disorders_PrePreg": "Prior_Mental_Disorder",
    "Diseases_of_the_nervous_system_PrePreg": "Prior_Nervous_System_Disease",
    "Diseases_of_the_eye_and_adnexa_PrePreg": "Prior_Eye_Disease",
    "Diseases_of_the_ear_and_mastoid_process_PrePreg": "Prior_Ear_Disease",
    "Diseases_of_the_circulatory_system_PrePreg": "Prior_Circulatory_Disease",
    "Diseases_of_the_respiratory_system_PrePreg": "Prior_Respiratory_Disease",
    "Diseases_of_the_digestive_system_PrePreg": "Prior_Gastrointestinal_Disease",
    "Diseases_of_the_skin_and_subcutaneous_tissue_PrePreg": "Prior_Skin_Disease",
    "Diseases_of_the_musculoskeletal_system_and_connective_tissue_PrePreg": "Prior_Musculoskeletal_Disease",
    "Diseases_of_the_genitourinary_system_PrePreg": "Prior_Genitourinary_Disease",
    "Pregnancy_with_abortive_outcome_PrePreg": "Prior_Aborted_Pregnancy",
    "Other_maternal_disorders_predominantly_related_to_pregnancy_PrePreg": "Prior_Other_Pregnancy_Related_Disorder",
    "Maternal_infectious_and_parasitic_diseases_classifiable_elsewhere_but_complicating_pregnancy_childbirth_and_the_puerperium_PrePreg": "Prior_Pregnancy_Related_Infectious_Disease",
    "Certain_conditions_originating_in_the_perinatal_period_PrePreg": "Prior_Perinatal_Conditions",
    "Congenital_malformations_deformations_and_chromosomal_abnormalities_PrePreg": "Prior_Congenital_Abnormality",
    "Symptoms_signs_and_abnormal_clinical_and_laboratory_findings_not_elsewhere_classified_PrePreg": "Prior_Other_Symptoms_or_Findings",
    "Injury_poisoning_and_certain_other_consequences_of_external_causes_PrePreg": "Prior_Injury_or_Poisoning",
    "Resistance_to_antimicrobial_and_antineoplastic_drugs_PrePreg": "Prior_AMR",
    "External_causes_of_morbidity_and_mortality_PrePreg": "Prior_External_Cause_MM",
    "Factors_influencing_health_status_and_contact_with_health_services_PrePreg": "Prior_Other_Factors_Influencing_Health_or_Access",

    "Certain_infectious_and_parasitic_diseases_DuringPreg": "Infectious_Disease_this_pregnancy",
    "Neoplasms_DuringPreg": "Neoplasm_this_pregnancy",
    "Diseases_of_the_blood_and_blood_forming_organs_and_certain_disorders_involving_the_immune_mechanism_DuringPreg": "Blood_or_Immune_Disease_this_pregnancy",
    "Endocrine_nutritional_and_metabolic_diseases_DuringPreg": "Endocrine_or_Metabolic_Disease_this_pregnancy",
    "Mental_and_behavioural_disorders_DuringPreg": "Mental_Disorder_this_pregnancy",
    "Diseases_of_the_nervous_system_DuringPreg": "Nervous_System_Disease_this_pregnancy",
    "Diseases_of_the_eye_and_adnexa_DuringPreg": "Eye_Disease_this_pregnancy",
    "Diseases_of_the_ear_and_mastoid_process_DuringPreg": "Ear_Disease_this_pregnancy",
    "Diseases_of_the_circulatory_system_DuringPreg": "Circulatory_Disease_this_pregnancy",
    "Diseases_of_the_respiratory_system_DuringPreg": "Respiratory_Disease_this_pregnancy",
    "Diseases_of_the_digestive_system_DuringPreg": "Gastrointestinal_Disease_this_pregnancy",
    "Diseases_of_the_skin_and_subcutaneous_tissue_DuringPreg": "Skin_Disease_this_pregnancy",
    "Diseases_of_the_musculoskeletal_system_and_connective_tissue_DuringPreg": "Musculoskeletal_Disease_this_pregnancy",
    "Diseases_of_the_genitourinary_system_DuringPreg": "Genitourinary_Disease_this_pregnancy",
    "Pregnancy_with_abortive_outcome_DuringPreg": "Abortive_Outcome",
    "Other_maternal_disorders_predominantly_related_to_pregnancy_DuringPreg": "Other_Pregnancy_Related_Disorder_this_pregnancy",
    "Maternal_infectious_and_parasitic_diseases_classifiable_elsewhere_but_complicating_pregnancy_childbirth_and_the_puerperium_DuringPreg": "Pregnancy_Related_Infectious_Disease_this_pregnancy",
    "Certain_conditions_originating_in_the_perinatal_period_DuringPreg": "Perinatal_Conditions_this_pregnancy",
    "Congenital_malformations_deformations_and_chromosomal_abnormalities_DuringPreg": "Congenital_Abnormality_this_pregnancy",
    "Symptoms_signs_and_abnormal_clinical_and_laboratory_findings_not_elsewhere_classified_DuringPreg": "Other_Symptoms_or_Findings_this_pregnancy",
    "Injury_poisoning_and_certain_other_consequences_of_external_causes_DuringPreg": "Injury_or_Poisoning_this_pregnancy",
    "Resistance_to_antimicrobial_and_antineoplastic_drugs_DuringPreg": "AMR_this_pregnancy",
    "External_causes_of_morbidity_and_mortality_DuringPreg": "External_Cause_MM_this_pregnancy",
    "Factors_influencing_health_status_and_contact_with_health_services_DuringPreg": "Other_Factors_Influencing_Health_or_Access_this_pregnancy",
})


In [0]:
write_table_name = "all_datasets_diagnoses_stage"

df_all_diags.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")